# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load Croissant metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access name and description (via dataset.metadata attributes)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Inspect available record sets, their fields, and IDs.

Every Croissant entity (record set, field, column) has a unique `@id`. We'll enumerate the available record sets, and then explore the fields and columns for each.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.record_sets
print(f"Total Record Sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs['@id']}")
    print(f"    name: {rs.get('name', 'N/A')}")
    print(f"    description: {rs.get('description', 'N/A')}")
    # List the fields/columns in this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # just one field
        fields = [fields]
    print(f"    Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"      - @id: {f.get('@id', 'N/A')} name: {f.get('name', 'N/A')}")
        else:
            print(f"      - @id: {f}")
    print()

### Example Data Preview
Let's print several records with field values referencing by `@id` for one record set, as an example overview.

In [ ]:
# Preview first records for the main record set
# Select the main record set by @id (choose index 0 from previous cell if unsure)
main_record_set_id = record_sets[0]['@id'] if record_sets else None
print(f"Preview of first 3 records in {main_record_set_id}:")
for i, row in enumerate(dataset.records(record_set=main_record_set_id)):
    print(row)
    if i >= 2:
        break

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame. This makes analysis easier and supports direct reference to field/column `@id`s as DataFrame columns.

All references to fields must be by `@id`.

In [ ]:
# For this dataset, collect data from all record sets. Use their @ids as keys.
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records from RecordSet @id: {rs_id}")
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"DataFrame columns for {rs_id} (by field @id):\n{df.columns.tolist()}\n")

### Quick Head of Main DataFrame
Let's look at the head of the main record set DataFrame.

In [ ]:
# Show head of main data table
if main_record_set_id and main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will:
- Select a numeric field (referenced by its `@id`), e.g. coverage or abundance fields.
- Apply filtering and create a new normalized field.
- (Optionally) group data by a categorical field (again, by `@id`).

In [ ]:
# Identify and select a numeric field by @id for demonstration.
df = dataframes[main_record_set_id]
print('Available columns (@id):', df.columns.tolist())

# Example: select first numeric column for demonstration purposes
import numpy as np
numeric_field_id = None
for c in df.columns:
    # Attempt to see if column data is numeric
    try:
        vals = pd.to_numeric(df[c], errors='coerce')
        if np.isfinite(vals).sum() > 0:
            numeric_field_id = c
            break
    except Exception:
        continue
if numeric_field_id is None:
    raise ValueError("No numeric field found in the main record set.")

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Convert column to numeric (in case it comes as string)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Outlier removal: filter values > lower threshold (pick mean/2 as threshold here)
threshold = df[numeric_field_id].mean()/2
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Number of records after filtering {numeric_field_id} > {threshold:.3f}: {len(filtered_df)}")

# Normalize the field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, if present
group_field = None
for c in df.columns:
    if c != numeric_field_id and df[c].dtype == object and df[c].nunique() > 1:
        group_field = c
        break
if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame("mean").reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field}:")
    print(grouped.head())

## 5. Visualization
Visualize the numeric field distribution (histogram) and grouped means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field exists, plot grouped bar
if group_field:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped, x=group_field, y='mean')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"{numeric_field_id} (mean)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded FAIR² mass spectrometry protein data using `mlcroissant`, referencing all entities by their Croissant `@id`s.
- Explored and filtered protein-related numeric data fields for basic EDA.
- Summarized and visualized data distribution and groupwise means using identified fields.

You can extend this workflow for custom analyses or feature engineering as required. For detailed schema, refer to [`mlcroissant` docs](https://github.com/mlcommons/croissant).